# Feature Engineering with Pandas

## Definition
**Feature engineering** is the process of creating new, more informative input features from raw data
to improve ML model performance. Pandas is the primary tool for this.

### Common Techniques
| Technique | Description |
|-----------|-------------|
| Interaction features | `col_a * col_b`, `col_a / col_b` |
| Polynomial features | `col ** 2`, `np.sqrt(col)` |
| Binning | `pd.cut()`, `pd.qcut()` |
| Aggregation features | Group means, stds, counts |
| Lag/rolling features | Time-series lookback |
| Target encoding | Replace category with target stat |
| Date features | Year, month, dayofweek from timestamp |
| Text features | Length, word count, regex extracts |
| Ratio features | A/B, A/(A+B) |

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
df = pd.DataFrame({
    'user_id': range(200),
    'age': rng.integers(18, 65, 200),
    'income': rng.normal(60000, 20000, 200).clip(0),
    'spend': rng.normal(2000, 800, 200).clip(0),
    'n_transactions': rng.integers(1, 50, 200),
    'account_age_days': rng.integers(30, 3650, 200),
    'city': rng.choice(['NYC', 'LA', 'Chicago', 'Houston'], 200),
    'gender': rng.choice(['M', 'F'], 200),
    'churn': rng.choice([0, 1], 200, p=[0.8, 0.2])
})
print(df.shape)
print(df.head(3))

In [ ]:
# 1. Ratio and interaction features
df['spend_to_income_ratio'] = df['spend'] / (df['income'] + 1)
df['avg_transaction_value'] = df['spend'] / (df['n_transactions'] + 1)
df['income_per_day'] = df['income'] / 365
df['spend_per_account_day'] = df['spend'] / df['account_age_days']

print('Ratio features created:')
print(df[['income', 'spend', 'spend_to_income_ratio', 'avg_transaction_value']].head())

In [ ]:
# 2. Polynomial / mathematical transforms
df['log_income'] = np.log1p(df['income'])
df['sqrt_n_transactions'] = np.sqrt(df['n_transactions'])
df['age_squared'] = df['age'] ** 2
df['account_age_years'] = df['account_age_days'] / 365

print('Transform features:')
print(df[['income', 'log_income', 'n_transactions', 'sqrt_n_transactions']].head())

In [ ]:
# 3. Binning (age groups, income brackets)
df['age_group'] = pd.cut(df['age'], bins=[18, 25, 35, 50, 65],
    labels=['18-25', '26-35', '36-50', '51-65'])

df['income_bracket'] = pd.qcut(df['income'], q=4,
    labels=['Low', 'Med-Low', 'Med-High', 'High'])

print('Binned features:')
print(df[['age', 'age_group', 'income', 'income_bracket']].head(8))

In [ ]:
# 4. Aggregation features (group statistics)
city_stats = df.groupby('city').agg(
    city_mean_income=('income', 'mean'),
    city_mean_spend=('spend', 'mean'),
    city_churn_rate=('churn', 'mean')
).reset_index()

df = df.merge(city_stats, on='city', how='left')
print('City aggregation features:')
print(df[['user_id', 'city', 'income', 'city_mean_income', 'city_churn_rate']].head())

In [ ]:
# 5. Flag / indicator features
df['is_high_spender'] = (df['spend'] > df['spend'].quantile(0.75)).astype(int)
df['is_senior'] = (df['age'] >= 50).astype(int)
df['is_long_tenure'] = (df['account_age_days'] > 365).astype(int)
df['has_many_transactions'] = (df['n_transactions'] > 20).astype(int)

print('Flag features:')
print(df[['user_id', 'is_high_spender', 'is_senior', 'is_long_tenure']].head())

In [ ]:
# 6. Cross features (interaction between two categories)
df['city_gender'] = df['city'] + '_' + df['gender']
print('City-gender cross feature:', df['city_gender'].value_counts().head(5))

In [ ]:
# 7. Rank features (percentile within group)
df['income_rank'] = df['income'].rank(pct=True)
df['income_rank_in_city'] = df.groupby('city')['income'].rank(pct=True)

print('Rank features:')
print(df[['user_id', 'city', 'income', 'income_rank', 'income_rank_in_city']].head())